# **基于静态 Tensor 框架的 SIMD 编程**

## 概述

本小节介绍 Ascend C SIMD API 中的**静态 Tensor 编程模式**。静态 Tensor 是一种面向 C++ Tensor 的算子编程方式：开发者直接使用 `GlobalTensor`、`LocalTensor` 和 `LocalMemAllocator` 描述 GM、本地存储与计算数据，并显式规划数据搬运、流水同步和 buffer 复用。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 了解 Ascend C 算子中 Host 侧启动 kernel、Device 侧访问 GM 的基本流程；
- 理解 `CopyIn → Compute → CopyOut` 三段式的基本算子结构；
- 了解芯片架构，理解 GM、UB、L1、L0A、L0B、L0C 等 buffer 在硬件上的位置和关系。

### 学习目标

完成本小节后，开发者应能够：

1. 理解静态 Tensor 面向硬件存储与流水显式编程的特点；
2. 掌握 `GlobalTensor`、`LocalMemAllocator`、`LocalTensor` 的基本使用方式；
3. 理解为什么需要核内同步，并正确使用 `PipeBarrier` 和 `SetFlag/WaitFlag` 表达流水依赖；
4. 理解 `DataCopy` 的连续搬运、对齐约束和非对齐搬运场景；
5. 初步理解基于 `RegBase` 的编程；
6. 能够读懂一个基础静态 Tensor Add 算子的实现结构。

### 本节内容

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">章节</th>
      <th align="left">内容</th>
      <th align="left">学习期望</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>基本特点</td>
      <td>梳理静态 Tensor 中 GM、Local Memory、计算流水之间的数据流</td>
      <td>建立整体编程流程</td>
    </tr>
    <tr>
      <td>内存分配方式</td>
      <td>介绍 <code>GlobalTensor</code>、<code>LocalMemAllocator</code> 与各级硬件存储</td>
      <td>掌握 buffer 规划方法</td>
    </tr>
    <tr>
      <td>流水同步</td>
      <td>说明 AI Core 并行流水、<code>PipeBarrier</code>、<code>SetFlag/WaitFlag</code> 的使用场景</td>
      <td>能判断同步点保护的数据依赖</td>
    </tr>
    <tr>
      <td>数据搬入</td>
      <td>说明连续搬运、对齐搬运、非对齐搬运与 GM 到 UB 的搬入方式</td>
      <td>能写出基础 <code>DataCopy</code> 代码</td>
    </tr>
    <tr>
      <td>计算</td>
      <td>介绍静态 Tensor 基础 API 与 RegBase/vf call 的入门写法</td>
      <td>能理解寄存器级向量计算路径</td>
    </tr>
    <tr>
      <td>数据搬出</td>
      <td>说明 UB 到 GM 的结果写回和复用前同步</td>
      <td>能判断写回与 buffer 复用时机</td>
    </tr>
    <tr>
      <td>算子实例</td>
      <td>以静态 Tensor Add 算子串联初始化、搬入、计算、搬出</td>
      <td>能读懂完整基础样例</td>
    </tr>
  </tbody>
</table>


---
# 1. 静态 Tensor 的基本特点

静态 Tensor 编程仍然遵循基础算子的三段式流程：

```cpp
CopyIn();   // 数据搬运
Compute();  // 本地计算
CopyOut();  // 数据搬出
```

与更高层封装不同，静态 Tensor 代码需要直接描述每个阶段使用哪块本地存储、由哪条硬件流水写入、由哪条硬件流水读取，以及什么时候可以复用 buffer。开发者需要把 Tensor 的物理位置、数据依赖和同步关系显式写在代码中。

静态 Tensor 的基本计算流程如下（以 Vector 算子为例）：

![静态 Tensor 基本计算流程](images/03_03_07_static_tensor/1_basic_compute_flow.svg)

主要流程如下：

1. 使用 `DataCopy`（MTE2 流水）将输入数据从 GM 搬运到 UB 的输入 Tensor；
2. 在 Vector 读取输入 Tensor 前，等待 MTE2 搬入流水完成；
3. Vector 单元读取 UB 输入 Tensor，完成计算并将结果写回 UB 输出 Tensor；
4. 在 MTE3 搬出输出 Tensor 前，等待 Vector 计算流水完成；
5. 使用 `DataCopy`（MTE3 流水）将 UB 输出 Tensor 的结果写回 GM。

主要特征是：**Tensor 物理位置、同步关系和 buffer 复用时机需要在代码中显式指定**。因此静态 Tensor 更适合需要精细控制本地存储和硬件流水的计算场景。

静态 Tensor kernel 还需要在`__global__`函数入口和退出位置完成基础运行状态处理：

```cpp
AscendC::InitSocState();
// kernel主体：GM绑定、本地Tensor分配、搬运、计算、写回
AscendC::PipeBarrier<PIPE_ALL>();
```

`InitSocState()` 用于初始化算子执行所需的 SoC 状态、寄存器和流水相关上下文；kernel 结束前的 `PipeBarrier<PIPE_ALL>()` 用于等待已发出的流水指令完成，避免 kernel 退出时仍有未完成的本地计算或搬运操作。


---
# 2. 内存分配

静态 Tensor 编程中，Buffer内存需要手动分配在硬件位置。主要有两种方式：

## 2.1 通过`LocalMemAllocator`分配

这种分配方式需要根据开发根据Tensor存储的物理位置，去创建对应的`LocalMemAllocator`对象。

再通过`Alloc`方法，创建对应物理位置的LocalTensor去存储和计算数据。

```cpp
// tileLen非静态，LocalMemAllocator分配Buffer空间场景
AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator; // 创建UB位置的LocalMemAllocator
AscendC::LocalTensor<half> xLocal = ubAllocator.Alloc<half>(tileLen); // 创建元素个数为tileLen的LocalTensor
AscendC::LocalTensor<half> yLocal = ubAllocator.Alloc<half>(tileLen); // 创建元素个数为tileLen的LocalTensor
AscendC::LocalTensor<half> zLocal = ubAllocator.Alloc<half>(tileLen); // 创建元素个数为tileLen的LocalTensor
```

以下是常用的`LocalMemAllocator`对象及其对应的位置：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">对象</th>
      <th align="left">常用场景</th>
      <th align="left">物理位置</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::UB&gt;</code></td>
      <td>在 UB 上分配 Vector 计算使用的本地 buffer</td>
      <td>Unified Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L1&gt;</code></td>
      <td>在 L1 上分配 Cube 前级搬入 buffer</td>
      <td>L1 Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L0A&gt;</code></td>
      <td>在 L0A 上分配矩阵 A 分形数据</td>
      <td>L0A Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L0B&gt;</code></td>
      <td>在 L0B 上分配矩阵 B 分形数据</td>
      <td>L0B Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::BIAS&gt;</code></td>
      <td>在 BiasTable Buffer 上分配 Bias 数据</td>
      <td>BT Buffer</td>
    </tr>
    <tr>
      <td><code>AscendC::LocalMemAllocator&lt;AscendC::Hardware::L0C&gt;</code></td>
      <td>在 L0C 上分配 Cube 累加结果</td>
      <td>L0C Buffer</td>
    </tr>
  </tbody>
</table>


## 2.2 通过`LocalTensor`构造函数分配

这种分配方式直接通过构造函数进行内存空间的指定。

```cpp
// tileLen非静态，构造函数分配Buffer空间场景
AscendC::LocalTensor<half> xLocal(AscendC::TPosition::VECCALC, xAddr, tileLen); // 在UB上创建基地址为xAddr，元素个数为tileLen的LocalTensor
AscendC::LocalTensor<half> yLocal(AscendC::TPosition::VECCALC, yAddr, tileLen); // 在UB上创建基地址为yAddr，元素个数为tileLen的LocalTensor
AscendC::LocalTensor<half> zLocal(AscendC::TPosition::VECCALC, zAddr, tileLen); // 在UB上创建基地址为zAddr，元素个数为tileLen的LocalTensor
```

`TPosition`与`Hardware`的对应关系请参考文章[SIMD-API 通用说明和约束](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%80%9A%E7%94%A8%E8%AF%B4%E6%98%8E%E5%92%8C%E7%BA%A6%E6%9D%9F.md)。


> `GlobalTensor` 不负责申请 GM 内存，它通常在 kernel 中绑定 Host 侧传入的 GM 地址。

---
# 3. 流水同步

AI Core 内部存在多条可以并行发射的硬件指令流水队列。以基础 Vector 场景为例，一轮计算通常涉及三类流水：

- `MTE2`：从 GM 搬入数据到 UB；
- `V`：在 UB 或寄存器上执行 Vector 计算；
- `MTE3`：将 UB 输出结果搬回 GM。

这些流水可以异步并行运行。并行能够提升吞吐，但当不同流水读写同一片 Local Memory 时，单纯依赖代码先后顺序并不能保证硬件执行时序。例如，代码中先写了 `DataCopy(GM → UB)`，再写 `Add`，并不意味着 Vector 读取 UB 时 MTE2 一定已经完成搬入；同理，代码中先写 `Add`，再写 `DataCopy(UB → GM)`，也不意味着 MTE3 搬出时 Vector 一定已经完成写入。

![MTE2 / V / MTE3 流水同步关系](images/03_03_07_static_tensor/3_pipeline_sync.svg)

因此，静态 Tensor 中的同步需要围绕真实数据依赖来判断：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">依赖类型</th>
      <th align="left">典型场景</th>
      <th align="left">同步目标</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>跨流水读后写/写后读</td>
      <td>MTE2 写 UB 后，Vector 读 UB</td>
      <td>读之前等待数据搬运完成</td>
    </tr>
    <tr>
      <td>跨流水写后读/写后写</td>
      <td>Vector 写 UB 后，MTE3 读 UB；MTE3 读完后下一轮复用 UB</td>
      <td>复用或搬出前等待前序流水完成</td>
    </tr>
    <tr>
      <td>同一流水内部依赖</td>
      <td>后一条 Vector 指令依赖前一条 Vector 指令结果</td>
      <td>后序指令开始前等待前序指令读写完成</td>
    </tr>
  </tbody>
</table>

开发者可以使用两类接口插入同步：

1. `PipeBarrier<PIPE_*>`：约束同一类流水内部的指令顺序；
2. `SetFlag/WaitFlag`：约束不同硬件流水之间的指令顺序。


## 3.1 `PipeBarrier`：同一流水内部的顺序依赖

`PipeBarrier` 用于约束同一类流水内的指令顺序。同一流水中的指令虽然按顺序发射，但后一条指令开始执行时，前一条指令的数据读写不一定已经全部完成；如果后一条指令依赖前一条指令写出的结果，就需要插入对应流水的 `PipeBarrier`。

以 Vector 计算为例：

```cpp
AscendC::Mul(tmpLocal, xLocal, xLocal, tileLen);    // tmp = x * x
AscendC::PipeBarrier<PIPE_V>();
AscendC::Add(zLocal, tmpLocal, yLocal, tileLen);    // z = tmp + y
```

这里的屏障保护的是 `tmpLocal` 上的中间结果。如果没有 `PipeBarrier<PIPE_V>()`，后一条 Vector 指令可能在前一条指令结果尚未完全可见时读取 `tmpLocal`。

`PipeBarrier` 的使用原则是：当同一流水中的后一条指令依赖前一条指令写出的结果时，需要显式保证顺序；如果两条指令之间没有数据依赖，过度插入屏障会降低流水并行度。


## 3.2 `SetFlag/WaitFlag`：跨流水同步

`SetFlag/WaitFlag` 用于表达不同硬件流水之间的同步。它本质上是在源流水和目的流水之间建立一组“锁”：

- `SetFlag`：源流水执行到该指令时，会等待源流水前序指令的读写操作完成，然后设置对应事件标志；
- `WaitFlag`：目的流水执行到该指令时，如果事件标志尚未设置，会阻塞目的流水后续指令；事件满足后清除标志并继续执行。

基础 Add 样例中的跨流水同步如下：

```cpp
constexpr event_t eventMte2ToV = EVENT_ID0;
constexpr event_t eventVToMte3 = EVENT_ID1;

AscendC::DataCopy(xLocal, xGm[offset], tileLen);
AscendC::DataCopy(yLocal, yGm[offset], tileLen);
AscendC::SetFlag<HardEvent::MTE2_V>(eventMte2ToV);
AscendC::WaitFlag<HardEvent::MTE2_V>(eventMte2ToV);

AscendC::Add(zLocal, xLocal, yLocal, tileLen);
AscendC::SetFlag<HardEvent::V_MTE3>(eventVToMte3);
AscendC::WaitFlag<HardEvent::V_MTE3>(eventVToMte3);

AscendC::DataCopy(zGm[offset], zLocal, tileLen);
```

这两个事件分别保护两段依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">数据流</th>
      <th align="left">保护的数据</th>
      <th align="left">缺失风险</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_V</code></td>
      <td><code>DataCopy</code> 搬入 → Vector 计算</td>
      <td>UB 输入 <code>xLocal</code>、<code>yLocal</code></td>
      <td>Vector 可能读到尚未搬完的数据</td>
    </tr>
    <tr>
      <td><code>V_MTE3</code></td>
      <td>Vector 计算 → <code>DataCopy</code> 写回</td>
      <td>UB 输出 <code>zLocal</code></td>
      <td>MTE3 可能写回尚未计算完成的数据</td>
    </tr>
  </tbody>
</table>

如果下一轮循环要复用 `xLocal`、`yLocal` 或 `zLocal`，还需要确认上一轮搬入、计算和搬出对这些 buffer 的访问已经结束。例如复用输出 `zLocal` 前，需要保证上一轮 `MTE3` 已经完成对 `zLocal` 的读取。


---
# 4. 数据搬入

数据搬入阶段负责将 GM 输入搬到本地存储。基础 Vector 静态 Tensor 通常将输入搬入 UB：

```cpp
AscendC::DataCopy(xLocal, xGm[tileOffset], tileLen);
```

`DataCopy` 最常见的形态是连续搬运：源地址和目的地址各自连续存放，`count` 参数表示搬运的元素个数，接口会根据数据类型换算成字节数。基础 Add 样例中的输入向量就是连续向量，适合使用这种最简单的搬运方式。

数据搬运时需要关注地址和长度是否满足硬件对齐要求。常见规则可以按以下方式理解：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">场景</th>
      <th align="left">推荐接口</th>
      <th align="left">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>连续且满足对齐要求</td>
      <td><code>DataCopy(dstLocal, srcGm, count)</code></td>
      <td>最基础、开销最低，适合本节 Add 样例</td>
    </tr>
    <tr>
      <td>多段连续块，块间存在固定间隔</td>
      <td>带 <code>DataCopyParams</code> / <code>DataCopyExtParams</code> 的 <code>DataCopy</code></td>
      <td>用 blockLen、blockCount、stride 描述分块搬运</td>
    </tr>
    <tr>
      <td>尾块或地址不满足对齐要求</td>
      <td><code>DataCopyPad</code> 或非对齐搬运接口</td>
      <td>处理非 32B 整数倍、需要补齐或尾块搬运的场景</td>
    </tr>
    <tr>
      <td>UB 内部重排或中间数据搬运</td>
      <td><code>Copy</code></td>
      <td>在 UB 内不同区域之间搬运，支持 mask 和间隔控制</td>
    </tr>
  </tbody>
</table>

搬运指令发出后，需要根据后续指令类型插入同步。若后续由 Vector 计算读取 `xLocal`，需要使用 `MTE2_V` 方向的事件同步：

![alt text](images/03_03_07_static_tensor/4_pipeline_flow.svg)


在课程基础样例中，`tileLen` 应优先选择满足连续搬运对齐要求的长度；当真实算子存在尾块时，需要额外处理最后一轮长度和同步关系，避免 Vector 读取未定义的填充区域。


---
# 5. 计算

静态 Tensor 的计算阶段可以使用两类常见写法：

1. 直接基于 `LocalTensor` 调用AscendC基础API，例如 `Add`、`Mul`、`Exp`、`LeakyRelu`；
2. 通过 `asc_vf_call` 调用 `__simd_vf__` 函数，在 VF 函数中使用 `RegTensor` 完成寄存器级计算。

基础API：
```cpp
AscendC::Mul(tmpLocal, xLocal, xLocal, tileLen);
AscendC::PipeBarrier<PIPE_V>();
AscendC::Adds(yLocal, tmpLocal, scalarValue, tileLen);
```

RegBase 写法会把计算主体放到 `__simd_vf__` 函数中。kernel 侧先把 GM 数据搬入 UB，完成 `MTE2_V` 同步后，将 UB 物理地址转换为 `__ubuf__` 指针，再通过 `asc_vf_call` 进入 VF 函数执行 `LoadAlign → RegTensor 计算 → StoreAlign`。

## 5.1 RegBase 与 vf call 入门

RegBase 面向 AIV 侧的 SIMD Register File。普通 Memory 矢量计算通常是 `数据搬入 → Compute → 数据搬出`：矢量指令直接从 UB 读源操作数，计算后再写回 UB。若多个 Vector 计算连续串联，中间结果会反复落到 UB，容易增加 UB 读写带宽压力和 Bank 冲突。

RegBase 将这一段计算展开为 `数据搬入 → Load → Compute → Store → 数据搬出`。其中 `Load → Compute → Store` 放在 VF 函数中执行：数据先从 UB 搬入寄存器，多个寄存器计算可以连续消费中间结果，最后再写回 UB。

![RegBase 编程模型总体结构](images/03_03_07_static_tensor/5_1_regbase_compute_flow.svg)

> 说明：RegBase 能力面向支持 SIMD Register File 的硬件。当前官方文档中，`asc_vf_call` 和 Reg 矢量计算基础 API 主要面向 Ascend 950PR / Ascend 950DT。若目标硬件不支持 RegBase，应继续使用普通 AscendC 基础 API。

### 5.1.1 kernel 侧与 VF 函数侧分工

RegBase 编程通常分为 kernel 侧和 VF 函数侧两部分：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">位置</th>
      <th align="left">主要职责</th>
      <th align="left">典型接口</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>kernel / <code>__aicore__</code> 侧</td>
      <td>管理 GM、UB、tile 循环和跨流水同步；准备 VF 函数入参</td>
      <td><code>GlobalTensor</code>、<code>LocalTensor</code>、<code>DataCopy</code>、<code>SetFlag/WaitFlag</code>、<code>asc_vf_call</code></td>
    </tr>
    <tr>
      <td>VF / <code>__simd_vf__</code> 侧</td>
      <td>在寄存器域完成 <code>Load → Compute → Store</code></td>
      <td><code>Reg::LoadAlign</code>、<code>Reg::Add</code>、<code>Reg::StoreAlign</code>、<code>Reg::UpdateMask</code></td>
    </tr>
  </tbody>
</table>

基础调用示例如下：

1. kernel 侧使用 `LocalMemAllocator` 在 UB 上分配 `LocalTensor`，并完成 GM 到 UB 的 `DataCopy`；
2. kernel 侧等待 `MTE2_V`，保证 Vector/VF 侧读取 UB 前，MTE2 搬入已经完成；
3. kernel 侧从 `LocalTensor.GetPhyAddr()` 获取 UB 物理地址，并转换为 `__ubuf__` 指针；
4. kernel 侧通过 `asc_vf_call<...>(...)` 调用 `__simd_vf__` 函数；
5. VF 函数侧使用 `LoadAlign` 将 UB 数据加载到 `RegTensor`，调用 RegBase API 计算，再使用 `StoreAlign` 写回 UB；
6. kernel 侧等待 `V_MTE3` 后，再通过 `DataCopy` 将 UB 输出写回 GM。

需要注意，寄存器不能直接访问 GM。GM 数据进入 RegBase 计算前必须先搬到 UB；寄存器结果要写回 GM，也必须先 Store 到 UB，再由 MTE3 搬出到 GM。

### 5.1.2 执行域与调用约束

RegBase 使用函数标签区分执行域，编译期会检查调用关系：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">标签</th>
      <th align="left">角色</th>
      <th align="left">可调用对象</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>__aicore__</code></td>
      <td>Device 侧普通函数或类成员函数</td>
      <td><code>__aicore__</code> 函数，或通过 <code>asc_vf_call</code> 调用 VF 函数</td>
    </tr>
    <tr>
      <td><code>__simd_vf__</code></td>
      <td>VF 函数入口</td>
      <td><code>__simd_callee__</code> 修饰的 Reg 矢量 API 或 VF 内部子函数</td>
    </tr>
    <tr>
      <td><code>__simd_callee__</code></td>
      <td>VF 内部子函数 / Reg API</td>
      <td>其他 <code>__simd_callee__</code> 函数</td>
    </tr>
  </tbody>
</table>

`AscendC::Reg::*` 接口属于 VF 域使用的 Reg 矢量 API，不能在普通 `__aicore__` 代码中直接调用。外层 kernel 需要通过 `asc_vf_call` 进入 VF 函数。

`asc_vf_call` 的入口函数必须使用 `__simd_vf__` 修饰，且不建议写成类的非静态成员函数。传参也应保持简单，通常传递 `__ubuf__` 裸指针和基本标量；不要把结构体、数组或跨执行域引用作为 VF 入参。

### 5.1.3 RegBase 基础 API 对象

常见对象如下：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">寄存器对象</th>
      <th align="left">作用</th>
      <th align="left">常见用法</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>RegTensor&lt;T&gt;</code></td>
      <td>保存 VL 宽度的矢量数据，是计算接口的主要源/目的操作数</td>
      <td><code>RegTensor&lt;float&gt; srcReg;</code></td>
    </tr>
    <tr>
      <td><code>MaskReg</code></td>
      <td>控制哪些元素参与计算，也用于尾块处理</td>
      <td><code>UpdateMask&lt;T&gt;(remainCount)</code></td>
    </tr>
    <tr>
      <td><code>AddrReg</code></td>
      <td>保存 UB 访问偏移，适合固定 stride 或多维索引</td>
      <td><code>CreateAddrReg(...)</code></td>
    </tr>
    <tr>
      <td><code>UnalignRegForLoad</code></td>
      <td>辅助 UB 非对齐搬入</td>
      <td><code>LoadUnAlignPre</code> + <code>LoadUnAlign</code></td>
    </tr>
    <tr>
      <td><code>UnalignRegForStore</code></td>
      <td>辅助 UB 非对齐搬出</td>
      <td><code>StoreUnAlign</code> + <code>StoreUnAlignPost</code></td>
    </tr>
  </tbody>
</table>

本节只使用最基础的 `RegTensor<T>` 与 `MaskReg`。其中 `RegTensor` 作为寄存器整体使用，不能像数组一样按元素索引访问；`MaskReg` 用来描述本轮计算的有效元素，尤其适合处理最后一个不足完整 VL 的尾块。

### 5.1.4 使用 RegBase 实现 Add

下面代码片段展示使用 RegBase 实现 Add 的核心结构。样例为了突出 RegBase 调用链，只保留 VF 函数和 kernel 侧调用位置：

```cpp
template <typename T, uint32_t singleCoreLength>
__simd_vf__ inline void AddVF(__ubuf__ T* dstAddr, uint32_t count)
{
    constexpr uint32_t oneRepeatSize = AscendC::GetVecLen() / sizeof(T);
    uint32_t repeatTimes = AscendC::CeilDivision(singleCoreLength, oneRepeatSize);
    AscendC::Reg::MaskReg mask;
    AscendC::Reg::RegTensor<T> srcReg, dstReg;

    for (uint32_t i = 0; i < repeatTimes; ++i) {
        uint32_t remainCount = count - i * oneRepeatSize;
        uint32_t validCount = remainCount > oneRepeatSize ? oneRepeatSize : remainCount;
        mask = AscendC::Reg::UpdateMask<T>(validCount);
        AscendC::Reg::LoadAlign(srcReg, dstAddr + i * oneRepeatSize);
        AscendC::Reg::LoadAlign(dstReg, dstAddr + i * oneRepeatSize);
        AscendC::Reg::Add(dstReg, dstReg, srcReg, mask);
        AscendC::Reg::StoreAlign(dstAddr + i * oneRepeatSize, dstReg, mask);
    }
}
```

`oneRepeatSize` 表示一个 VF 指令按当前数据类型可处理的元素数。若 VL 为 256B，则 `float` 对应 64 个元素，`half` 对应 128 个元素。`repeatTimes` 覆盖当前 tile 或当前核负责的全部元素；每轮循环先计算本轮剩余元素，再通过 `UpdateMask<T>(validCount)` 生成有效元素掩码，尾块不会写出无效元素。

kernel 侧调用 VF 函数的位置如下：

```cpp
AscendC::DataCopy(yLocal, xGm[coreOffset], singleCoreLength);
AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);
AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);

__ubuf__ float* dstAddr = reinterpret_cast<__ubuf__ float*>(yLocal.GetPhyAddr());
asc_vf_call<AddVF<float, singleCoreLength>>(dstAddr, singleCoreLength);

AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
AscendC::DataCopy(yGm[coreOffset], yLocal, singleCoreLength);
```

### 5.1.5 同步与搬运选择

VF 函数退出时，函数内部的 Reg 矢量指令已经完成；但进入 VF 函数前后仍然要关注 UB 与其他硬件流水之间的交接：

- `DataCopy` 从 GM 搬入 UB 后，需要通过 `MTE2_V` 保证 VF 函数读取 UB 前数据已就绪；
- `StoreAlign` 将寄存器结果写回 UB 后，kernel 侧需要通过 `V_MTE3` 保证 MTE3 写回 GM 前结果已就绪；
- VF 函数内部若不同寄存器搬运访问同一段 UB 且存在写后读、写后写关系，需要插入 Reg 流水同步；同一寄存器上的写后读依赖通常由硬件按指令顺序保证。

搬运接口选择上，地址满足对齐要求时优先使用 `LoadAlign` / `StoreAlign`；地址不对齐时可使用通用 `Load` / `Store`，性能敏感场景再考虑 `LoadUnAlignPre`、`LoadUnAlign`、`StoreUnAlign`、`StoreUnAlignPost` 组合。基础课程先掌握对齐搬运路径即可。

### 5.1.6 RegBase使用场景

从 AscendC 基础 API 迁移到 RegBase 时，需要判断以下几点：

1. 目标硬件是否支持 RegBase；
2. 当前计算是否由多条连续 Vector 计算组成，且中间结果反复读写 UB；
3. 输入输出是否已经在 UB 上，且能以较规则的连续地址访问；
4. 是否能接受 VF 函数、mask、repeat、对齐搬运等额外复杂度。

RegBase 更适合用于连续 Vector 计算场景。


---
# 6. 数据搬出

数据搬出阶段负责将本地输出 buffer 写回 GM。基础 Vector 场景中，输出通常从 UB 写回 GM：

```cpp
AscendC::DataCopy(zGm[tileOffset], zLocal, tileLen);
```

写回前必须确认 `zLocal` 的计算流水已经完成。如果 `zLocal` 由 Vector 计算得到，则通常需要使用 `V_MTE3` 同步：

```cpp
AscendC::Add(zLocal, xLocal, yLocal, tileLen);
AscendC::SetFlag<HardEvent::V_MTE3>(eventVToMte3);
AscendC::WaitFlag<HardEvent::V_MTE3>(eventVToMte3);
AscendC::DataCopy(zGm[tileOffset], zLocal, tileLen);
```

搬出也要关注连续性和对齐。基础样例中 `zLocal` 与 `zGm[tileOffset]` 都按连续向量写回，适合直接使用 `DataCopy`。如果输出存在非连续布局、尾块不足一个对齐粒度、或需要补齐/格式转换，应选择对应的高维搬运、非对齐搬运或随路转换搬运接口。

如果下一轮循环要复用 `zLocal`，还需要确保 MTE3 已经完成对该 buffer 的读取。常见写法是在写回后增加 `MTE3_MTE2` 或其他与下一轮指令匹配的同步，保证下一轮重新写入 UB 前，上一轮搬出已经结束。


---
# 7. 算子实例：静态 Tensor Add 算子

本节以 Add 算子作为静态 Tensor 入门样例。计算逻辑为：

```text
Z = X + Y
```

### Kernel 入口

```cpp
template <uint32_t totalLength, uint32_t tileLength>
__global__ __vector__ void add_static_tensor(GM_ADDR x, GM_ADDR y, GM_ADDR z)
```

使用 `__vector__` 声明，表示该样例只在 Vector 侧运行。Host 侧负责申请 Device 内存、拷贝输入、启动 kernel，并将输出拷回 Host 进行校验。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">参数</th>
      <th align="left">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>x</code></td>
      <td>输入向量 X 的 GM 地址</td>
    </tr>
    <tr>
      <td><code>y</code></td>
      <td>输入向量 Y 的 GM 地址</td>
    </tr>
    <tr>
      <td><code>z</code></td>
      <td>输出向量 Z 的 GM 地址</td>
    </tr>
    <tr>
      <td><code>totalLength</code></td>
      <td>总元素数</td>
    </tr>
    <tr>
      <td><code>tileLength</code></td>
      <td>单轮处理的元素数</td>
    </tr>
  </tbody>
</table>

### 样例规格

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">项目</th>
      <th align="left">取值</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>输入 X</td>
      <td><code>float</code> 连续向量</td>
    </tr>
    <tr>
      <td>输入 Y</td>
      <td><code>float</code> 连续向量</td>
    </tr>
    <tr>
      <td>输出 Z</td>
      <td><code>float</code> 连续向量</td>
    </tr>
    <tr>
      <td>计算逻辑</td>
      <td><code>z = x + y</code></td>
    </tr>
    <tr>
      <td>本地存储</td>
      <td>UB</td>
    </tr>
    <tr>
      <td>主要流水</td>
      <td><code>MTE2 → V → MTE3</code></td>
    </tr>
  </tbody>
</table>

该样例参考 Ascend C SIMD API 的基础 [Add 样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/00_introduction/01_add/add)，并保留本课程使用的类封装方式，优化了同步类型。重点展示静态 Tensor 中输入 buffer 的分配、搬入、同步、计算和写回的完整流程。


## 7.1 kernel侧代码示例

下面代码展示基础静态 Tensor Add 算子的主体结构。为突出教学主线，示例只保留核心 Device 侧逻辑。

```cpp
#include "kernel_operator.h"

using namespace AscendC;

template <uint32_t totalLength, uint32_t tileLength>
class KernelAddStaticTensor {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z)
    {
        xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), totalLength);
        yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), totalLength);
        zGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(z), totalLength);
    }

    __aicore__ inline void Process()
    {
        LocalMemAllocator<Hardware::UB> ubAllocator;
        LocalTensor<float> xLocal = ubAllocator.Alloc<float, tileLength>();
        LocalTensor<float> yLocal = ubAllocator.Alloc<float, tileLength>();
        LocalTensor<float> zLocal = ubAllocator.Alloc<float, tileLength>();

        for (uint32_t offset = 0; offset < totalLength; offset += tileLength) {
            DataCopy(xLocal, xGm[offset], tileLength);
            DataCopy(yLocal, yGm[offset], tileLength);
            SetFlag<HardEvent::MTE2_V>(EVENT_ID0);
            WaitFlag<HardEvent::MTE2_V>(EVENT_ID0);

            Add(zLocal, xLocal, yLocal, tileLength);
            SetFlag<HardEvent::V_MTE3>(EVENT_ID0);
            WaitFlag<HardEvent::V_MTE3>(EVENT_ID0);

            DataCopy(zGm[offset], zLocal, tileLength);
            SetFlag<HardEvent::MTE3_MTE2>(EVENT_ID0);
            WaitFlag<HardEvent::MTE3_MTE2>(EVENT_ID0);
        }
    }

private:
    GlobalTensor<float> xGm;
    GlobalTensor<float> yGm;
    GlobalTensor<float> zGm;
};

template <uint32_t totalLength, uint32_t tileLength>
__global__ __vector__ void add_static_tensor(GM_ADDR x, GM_ADDR y, GM_ADDR z)
{
    AscendC::InitSocState();
    KernelAddStaticTensor<totalLength, tileLength> op;
    op.Init(x, y, z);
    op.Process();
    AscendC::PipeBarrier<PIPE_ALL>();
}
```


## 7.2 同步关系分析

该样例中有两个必要同步点：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">数据流</th>
      <th align="left">保护的数据</th>
      <th align="left">缺失风险</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_V</code></td>
      <td><code>DataCopy(xLocal, xGm[offset], tileLength)</code>、<code>DataCopy(yLocal, yGm[offset], tileLength)</code> → <code>Add</code></td>
      <td>UB 输入 <code>xLocal</code>、<code>yLocal</code></td>
      <td>Vector 可能读取未搬完的输入</td>
    </tr>
    <tr>
      <td><code>V_MTE3</code></td>
      <td><code>Add</code> → <code>DataCopy(zGm[offset], zLocal, tileLength)</code></td>
      <td>UB 输出 <code>zLocal</code></td>
      <td>MTE3 可能写回未计算完成的输出</td>
    </tr>
  </tbody>
</table>

有一个容易忽略的同步点：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">数据流</th>
      <th align="left">保护的数据</th>
      <th align="left">缺失风险</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE3_MTE2</code></td>
      <td><code>DataCopy(zGm[offset], zLocal, tileLength)</code> → 下一轮 <code>DataCopy(xLocal/yLocal, ..., tileLength)</code></td>
      <td>上一轮循环的 UB 输出 <code>zLocal</code></td>
      <td>下一轮复用 UB buffer 时，MTE3 可能仍在读取上一轮输出</td>
    </tr>
  </tbody>
</table>


---
# 8. 小结

静态 Tensor 编程更加贴合硬件执行规则，主要特点如下：

- 使用 `GlobalTensor` 描述 GM 输入输出，使用 `LocalMemAllocator` 在 UB、L1、L0 等硬件存储上显式分配 `LocalTensor`；
- 使用 `DataCopy`、`Copy`、`DataCopyPad` 等搬运接口描述 GM、UB 和其他本地存储之间的数据移动，并根据连续性、对齐要求和尾块形态选择合适接口；
- 使用 `PipeBarrier`、`SetFlag/WaitFlag` 明确表达同流水顺序依赖和跨流水指令依赖；
- 算子入口需要添加 `AscendC::InitSocState()` 初始化算子执行所需的寄存器和流水状态，算子退出前需要添加 `AscendC::PipeBarrier<PIPE_ALL>()` 等待已发出的流水操作完成；
- 在需要进一步减少 UB 中间结果读写时，可以将连续 Vector 计算迁移到 RegBase/vf call 路径。

学习静态 Tensor 时，应始终从“数据在哪个硬件存储上、由哪条指令流水读写、何时可以复用”这三个问题出发分析代码。


---
# 9. 课后练习

本节练习用于检查静态 Tensor 基础概念

## 题目 1：选择题

静态 Tensor 写法中，通常通过哪个对象在 UB 上分配本地 `LocalTensor`？

A. `GlobalTensor<half>`  
B. `LocalMemAllocator<AscendC::Hardware::UB>`  
C. `aclrtMalloc`  
D. `aclrtMemcpy`

## 题目 2：填空题

在 Device 侧，`__________` 用于绑定 GM 输入输出地址，并提供访问全局内存的 Tensor 视角。

## 题目 3：选择题

`MTE2_V` 同步点主要用于保护哪一段依赖？

A. MTE2 将 GM 数据搬入 UB 后，Vector 才能读取该 UB buffer  
B. Vector 计算完成后，MTE3 才能写回 GM  
C. Cube 计算完成后，Fixpipe 才能写回 GM  
D. Host 拷贝输入到 Device 后，kernel 才能启动

## 题目 4：填空题

同一 Vector 流水中，如果后一条 Vector 指令依赖前一条 Vector 指令的结果，通常使用 `__________` 保证顺序。

## 题目 5：选择题

RegBase 入门计算路径中，正确的顺序是：

A. `StoreAlign → asc_vf_call → LoadAlign`  
B. `DataCopy → Fixpipe → StoreAlign`  
C. `kernel 调用 asc_vf_call → VF 函数内 LoadAlign → RegBase 计算 → StoreAlign`  
D. `SetGlobalBuffer → aclrtMalloc → aclrtMemcpy`

## 题目 6：选择题

当数据尾块不满足普通连续搬运的对齐要求时，应该优先考虑哪类接口或处理方式？

A. 忽略尾块，直接按完整 tile 读取  
B. `DataCopyPad` 或非对齐搬运接口  
C. 只插入 `PipeBarrier<PIPE_V>()`  
D. 将输出直接写到寄存器中


## 课后练习答案

答案保存在本课程目录的 `answer/03_03_07_static_tensor/answers.md` 中。执行下面代码单元可查看参考答案。


In [ ]:
from pathlib import Path

answer_path = Path("answer/03_03_07_static_tensor/answers.md")
print(answer_path.read_text(encoding="utf-8"))
